In [1]:
import threading
import queue
import time
import random

# 1. Initialize a Bounded Queue with a strict maximum limit of 5 slots
freshest_queue = queue.Queue(maxsize=5)
stop_pipeline = threading.Event()

def fast_producer():
    """
    PRODUCER: Generates data very quickly (every 0.05 seconds).
    If the queue fills up, it drops the item to avoid blocking itself.
    """
    print(f"[{threading.current_thread().name}] Fast producer active.")
    reading_id = 1

    while not stop_pipeline.is_set():
        mock_data = {"id": reading_id, "val": round(random.uniform(10.0, 90.0), 2)}

        try:
            # block=False means if the 5 slots are full, it won't freeze;
            # it instantly throws a queue.Full error
            freshest_queue.put(mock_data, block=False)
            print(f"[{threading.current_thread().name}] Sent packet #{reading_id}")
        except queue.Full:
            # Overwrite protection: If the queue is full, skip this frame
            print(f"[{threading.current_thread().name}] Bounded Queue FULL! Dropping packet #{reading_id} to protect real-time stream.")

        reading_id += 1
        time.sleep(0.05)  # Produces fast!

    print(f"[{threading.current_thread().name}] Producer stopped.")


def freshest_consumer():
    """
    CONSUMER: Processes data slowly (every 0.3 seconds)
    but FLUSHES the queue to always extract the freshest data packet.
    """
    print(f"[{threading.current_thread().name}] Freshest data consumer active.")

    while not stop_pipeline.is_set() or not freshest_queue.empty():
        try:
            # Step A: Blocking wait for the first available item on the belt
            data_packet = freshest_queue.get(timeout=0.1)

            # Step B: THE FLUSH LOOP!
            # Keep aggressively grabbing items using get_nowait() until the queue is completely empty.
            # This discards the old intermediate packets and jumps straight to the newest one.
            while True:
                try:
                    freshest_packet = freshest_queue.get_nowait()
                    freshest_queue.task_done() # Checkmark the old discarded packet
                    data_packet = freshest_packet # Overwrite with the newer packet!
                except queue.Empty:
                    # The queue is totally empty now, meaning data_packet holds the freshest item!
                    break

            # Step C: Process ONLY the freshest packet we found
            print(f"    🌟 [{threading.current_thread().name}] TARGET GRABBED! Processing freshest packet #{data_packet['id']}...")
            time.sleep(0.3)  # Slow physical processing/movement delay

            freshest_queue.task_done()

        except queue.Empty:
            continue

    print(f"[{threading.current_thread().name}] Consumer stopped.")

# --- Execution ---
prod_thread = threading.Thread(target=fast_producer, name="Fast_Producer")
cons_thread = threading.Thread(target=freshest_consumer, name="Freshest_Consumer")

prod_thread.start()
cons_thread.start()

# Let the simulation run for 1.5 seconds
time.sleep(1.5)

print("\n[Main] Stopping pipeline...")
stop_pipeline.set()

prod_thread.join()
cons_thread.join()

print("[Main] Simulation finished safely.")

[Fast_Producer] Fast producer active.
[Fast_Producer] Sent packet #1
[Freshest_Consumer] Freshest data consumer active.
    🌟 [Freshest_Consumer] TARGET GRABBED! Processing freshest packet #1...
[Fast_Producer] Sent packet #2
[Fast_Producer] Sent packet #3
[Fast_Producer] Sent packet #4
[Fast_Producer] Sent packet #5
[Fast_Producer] Sent packet #6
    🌟 [Freshest_Consumer] TARGET GRABBED! Processing freshest packet #6...
[Fast_Producer] Sent packet #7
[Fast_Producer] Sent packet #8
[Fast_Producer] Sent packet #9
[Fast_Producer] Sent packet #10
[Fast_Producer] Sent packet #11
[Fast_Producer] Bounded Queue FULL! Dropping packet #12 to protect real-time stream.
    🌟 [Freshest_Consumer] TARGET GRABBED! Processing freshest packet #11...
[Fast_Producer] Sent packet #13
[Fast_Producer] Sent packet #14
[Fast_Producer] Sent packet #15
[Fast_Producer] Sent packet #16
[Fast_Producer] Sent packet #17
[Fast_Producer] Bounded Queue FULL! Dropping packet #18 to protect real-time stream.
    🌟 [Fresh